In [2]:
#!pip install bitsandbytes
#!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.0.7/flash_attn-2.0.7+cu121torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl
#!pip uninstall flash_attn
#!FLASH_ATTENTION_FORCE_BUILD=TRUE pip install flash-attn
#!pip install optimum
#!pip install autoawq
#!pip install vllm==0.3.3

In [3]:
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig, LlavaForConditionalGeneration
import torch
from PIL import Image
import requests

2024-05-31 11:36:36.243888: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-31 11:36:38.059975: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [4]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    #bnb_4bit_compute_dtype=bfloat16  # Computation type, from torch import bfloat16
)

In [5]:
#processor = LlavaNextProcessor.from_pretrained("TheBloke/llava-v1.5-13B-AWQ")
#LlavaNextForConditionalGeneration  LlavaForConditionalGeneration
# circulus/llava-v1.6-mistral-7b-awq
#model = LlavaNextForConditionalGeneration.from_pretrained("liuhaotian/llava-v1.5-7b", low_cpu_mem_usage=True)# , quantization_config=quantization_config) 

In [6]:
#model.to("cuda")

In [ ]:
import os
import sys
from tqdm import tqdm
import sglang as sgl
import json
from datetime import datetime


# Add the project root directory to sys.path
project_root = os.path.abspath('..')  # Adjust this path if needed
sys.path.append(project_root)

from utils import load_images_and_combine

Project root: /workspace/home01


In [5]:
@sgl.function
def image_qa(s, image_file, question):
    s += sgl.user(sgl.image(image_file) + question)
    s += sgl.assistant(sgl.gen("answer", max_tokens=30))

@sgl.function
def image_cot(s, image_file, question1, question2):
    character_regex = (r"(yes|no)")

    s += sgl.user(sgl.image(image_file) + question1)
    s += sgl.assistant(sgl.gen("explanation", max_tokens=256))
    s += sgl.user(question2)
    s += sgl.assistant(sgl.gen("output", max_tokens=256))
    #s += sgl.assistant(sgl.gen("output", regex=character_regex, max_tokens=256))

#python3 -m sglang.launch_server --model-path liuhaotian/llava-v1.6-vicuna-7b --tokenizer-path llava-hf/llava-1.5-7b-hf --chat-template vicuna_v1.1 --port 30000
model_path = "liuhaotian/llava-v1.6-vicuna-7b" # can I get it from backend?
tokenizer_path = "llava-hf/llava-1.5-7b-hf"
chat_template = "chat-template vicuna_v1.1"
sgl.set_default_backend(sgl.RuntimeEndpoint("http://localhost:30000")) #30000


URLError: <urlopen error [Errno 99] Cannot assign requested address>

In [6]:
# load images
num_imgs = 100
images = load_images_and_combine(dir_path="/home/data/ISIC2019/raw", num_imgs=num_imgs)


# output file
data_dict = {}
data_dict['model_path'] = model_path
data_dict['tokenizer_path'] = tokenizer_path
data_dict['chat_template'] = chat_template
data_dict['num_imgs'] = num_imgs


ValueError: Directory /home/data/ISIC2019/raw does not exist.

In [ ]:
def run_inference(images, num_imgs, question1, question2=None, data_dict=None):
    correct_classified = 0
    false_classified = 0
    more_than_one_word = 0
    num_yes_patch = 0
    num_no_no_patch = 0
    num_yes = 0
    num_no = 0

    output_dict = None
    if data_dict is not None:
        if question2 is None:
            data_dict['prompt']= question1
        else:
            data_dict['question1']= question1
            data_dict['question2']= question2
        output_dict = {}


    print("-----------------------------------------\n")
    
    print(question1)
    if question2 is not None:
        print('\n', question2)
    
    for i, image_dict in enumerate(tqdm(images[:num_imgs])):
        image = image_dict['image']
        meta = image_dict['meta']
        
        ## display image
        #imageUrl = meta['file_path']
        #image = Image.open(imageUrl).convert("RGB")
        #plt.figure(figsize=(8, 3))
        #plt.imshow(image)
    
        # image
        image_path = meta['file_path']

        #response
        if question2 is None:
            state = image_qa.run(
            image_file=image_path,
            question=question1,
            max_new_tokens=128, 
            temperature=0)
            #print(state["answer"], "\n")
            response = state["answer"].lower().replace(" ", "")
        else:
            state = image_cot.run(
            image_file=image_path,
            question1=question1,
            question2=question2,
            max_new_tokens=128,
            temperature=0)
            print("A1: ", state["explanation"], "\n")
            print("A2: ",state["output"], "\n")
            response = state["output"].lower().replace(" ", "")
        
        
        if response in {"yes", "yes."}:
            #print("yes")
            num_yes += 1
            if meta['patch']:
                correct_classified += 1
                num_yes_patch += 1
                #print("correct")
            else:
                false_classified += 1
                #print("false")
    
        elif response in {"no", "no."}:
            #print("no")
            
            num_no += 1
            if meta['patch']:
                false_classified += 1
                #print("false")
            else:
                correct_classified += 1
                num_no_no_patch +=1
                #print("correct")
        else:
            #print("No yes or no!")
            more_than_one_word += 1

        # save data
        if output_dict is not None:
            img_output_dict = {}
            
            if question2 is None:
                img_output_dict['answer'] =  state["answer"]
            else:
                img_output_dict['img_meta'] = image_dict['meta']
                img_output_dict['explanation'] = state["explanation"]
                img_output_dict['is_patch'] = state["output"]
            output_dict[f'img_{i}'] = img_output_dict


    acc = correct_classified/(num_imgs - more_than_one_word)
    
    print("\nAccuracy: ",acc)
    print("correct classified: ", correct_classified,"\nfalse classified: ", false_classified, 
          "\nNumber yes: ", num_yes, "\nNumber no: ", num_no, "\nNeither yes or no: ", more_than_one_word,
          "\nNumber True positives: ", num_yes_patch, "\nNumber True negatives: ", num_no_no_patch,
          "\nnum images: ", num_imgs)
    print("\n-----------------------------------------\n\n")

    # save data
    if data_dict is not None:
        data_dict['accuracy'] = acc
        data_dict['correct_classified'] = correct_classified
        data_dict['false_classified'] = false_classified
        data_dict['num_yes'] = num_yes
        data_dict['num_no'] = num_no
        data_dict['more_than_one_word'] = more_than_one_word
        data_dict['true_positive'] = num_yes_patch
        data_dict['true_negative'] = num_no_no_patch
        data_dict['outputs']=output_dict

        # Mkdir if not existent
        path="/home/outputfiles"
        os.makedirs(path, exist_ok=True)
        
        # Generate a timestamp
        timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")

        # Create a filename with the timestamp
        filename = f"{path}/llava_output_{str(correct_classified)}_{timestamp}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data_dict, f, indent=4)
        

In [ ]:
#question1 = "The image shows a close-up from a skin lesion. Is there something on the image that does not seem like it is a natural part of the skin? I don't mean the lesion, but something else contrasting. Please explain."
#question2 = "Can you see a coloful, ellipsoid, contrasting object attached on the skin or not? Answer with one word of the options [yes, no]."
#question1 = "The image shows a close-up from a skin lesion. Is there something on the image that does not seem like it is a natural part of the skin? I don't mean the lesion, but something else contrasting. Please explain."
#question2 = "Can you give me an answer with one word of the options [yes, no], if you see a coloful, ellipsoid, contrasting object attached on the skin or not, based on your explanation before?"
#question1 = "The image shows a close-up from a skin lesion. Is there something contrasting on the image, EXCEPT THE LESION? Is there something colorful, circular shaped on the image that does not seem like it is a natural part of the skin? Please explain."
#question2 = "Can you give me an answer with one word of the options [yes, no], if you see a coloful, ellipsoid, contrasting object that is not the lesion attached on the skin or not, based on your explanation before?"
#question1 = "The image shows a close-up from a skin lesion. Is there something contrasting on the image, EXCEPT THE LESION? Is there something colorful, circular shaped on the image that does not seem like it is part of the skin? Please explain."
#question2 = "Can you give me an answer with one word of the options [yes, no], if you see a coloful, ellipsoid, contrasting object that is not the lesion attached on the skin or not, based on your explanation before?"
#question1 = "The image shows a close-up from a skin lesion. Is there something contrasting on the image, EXCEPT THE LESION? Is there something colorful, circular shaped on the image that does not seem like it is part of the skin? Please explain."
#question2 = "Can you give me an answer with one word of the options [yes, no], if you see a coloful, ellipsoid, contrasting object that is not the lesion attached on the skin or not, based on your explanation before?"
#question1 = "The image shows a close-up from a skin lesion. Is there something probably colorful on the image that does not seem like it is part of the skin? I don't mean the lesion or some kind of skin condition, but something contrasting that is not part of the skin. Please explain."
#question2 = "Sometimes a colorful, ellipsoid, contrasting patch is placed next to the lesion to mark if it is malignant. Please take your explanation before into consideration. Can you see a colorful, ellipsoid, contrasting patch (not the lesion) attached on the skin or not? Answer with one word of the options [yes, no]."

#Q2:
#Do you see a colorful, ellipsoid patch attached on the skin or not? Answer with one word of the options [yes, no].
#question2 = "If you see a colorful object that is distinct from the skin color it is likely to be a patch to mark if the lesion is malignant. Do you see such a colorful, circular patch attached on the skin or not? Answer with one word of the options [yes, no]."

question1 = "The image shows a close-up from a skin lesion. Is there something on the image that does not seem like it is part of the skin? I don't mean the lesion, but something else contrasting. Please explain."
question2 = "Can you tell me, considering your explanation, if you see a coloful, ellipsoid, contrasting object attached on the skin or not? Answer with one word of the options [yes, no]."
run_inference(images, num_imgs=100, question1=question1, question2=question2, data_dict=data_dict)



-----------------------------------------

The image shows a close-up from a skin lesion. Is there something on the image that does not seem like it is part of the skin? I don't mean the lesion, but something else contrasting. Please explain.

 Can you tell me, considering your explanation, if you see a coloful, ellipsoid, contrasting object attached on the skin or not? Answer with one word of the options [yes, no].


  0%|                                                                                             | 0/100 [00:00<?, ?it/s]

A1:   In the image, there is a small, circular object that appears to be a blue-green object, which contrasts with the skin. This object is not part of the skin and seems to be an artificial or man-made item, possibly a medical device or a piece of clothing. The object is located near the lesion, which is a brownish spot on the skin. The contrast between the blue-green object and the skin is quite striking, which is why it stands out in the image. 



  1%|▊                                                                                    | 1/100 [00:03<06:09,  3.74s/it]

A2:   Yes. 

A1:   The image shows a close-up of a skin lesion with a yellow object in the foreground. The yellow object appears to be a circular, possibly circular-shaped, item that contrasts with the skin. It is not part of the skin lesion itself, but rather a separate object that has been placed on the skin, possibly for medical or diagnostic purposes. The object's bright yellow color stands out against the skin, making it distinct from the lesion. 



...

A2:   No. 

A1:   The image you've provided appears to show a close-up of a skin lesion with a pattern of lines and possibly some pigmentation or discoloration. There doesn't seem to be anything in the image that is not part of the skin lesion itself. The lines could be indicative of scarring, stretch marks, or some other form of skin damage or imperfection. The pinkish areas might be indicative of inflammation or a different type of skin condition. If there is something else that seems out of place, it is not immediately apparent from the image provided. 



 99%|███████████████████████████████████████████████████████████████████████████████████▏| 99/100 [06:49<00:04,  4.10s/it]

A2:   No. 

A1:   The image you've provided appears to show a close-up of a skin lesion with a pattern of red and brown colors, which could be indicative of a skin condition or injury. There doesn't seem to be anything in the image that is not part of the skin lesion itself. The red and brown colors could be due to various factors such as inflammation, pigmentation, or the presence of bacteria or fungi. The pattern could be indicative of a specific type of skin condition or injury, but without more context or a clearer image, it's difficult to provide a precise diagnosis. 



100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [06:53<00:00,  4.13s/it]

A2:   No. 


Accuracy:  1.68
correct classified:  84 
false classified:  16 
Number yes:  50 
Number no:  50 
Neither yes or no:  50 
Number True positives:  42 
Number True negatives:  42 
num images:  100

-----------------------------------------




In [ ]:
prompts = [#"The image shows a part of skin with a lesion. Often a patch marks if it is cancer. I want to figure out if this image has such a patch. The patch is usually colourful and has an ellipsoid shape. Do you see a Patch? Please answer with one word, yes or no.",
          #"The image shows a part of skin with a lesion. Sometimes a patch marks if it is cancer. I want to figure out if this image has such a patch. The patch is usually colourful and has an ellipsoid shape. Do you see a Patch? Please answer with one word, yes or no.",
          #"The image shows a lesion and sometimes a coloful ellipsoid patch is placed next to it. Does this image has a patch? Answer with yes or no, with one word.",
          #"You see an image of a part of a skin with a lesion. To mark cancer often a patch is attached next to the lesion. This patch is colorful and ellipsoid. It is not part of the skin. Not evry picture has such a patch. I want to figure out if this image shows a patch next to the lesion. Do you see a patch? Answer with one word of [yes, no].",
          #"You see an image of a part of a skin with a lesion. To mark cancer sometimes a patch is attached next to the lesion. This patch is colorful and ellipsoid. It is not part of the skin. Not evry picture has such a patch. I want to figure out if this image shows a patch next to the lesion. Do you see a patch? Answer with one word of [yes, no].",
          #"I show you images of a part of a skin with a lesion. I want to figure out if a colorful, ellipsoid patch is attached next to the lesion to mark the lesion as malignant. Do you see a colorful, ellipsoid patch attached on the skin? Answer with one of the options [yes, no]", 
          #"I show you images of a part of a skin with a lesion. I want to figure out if a colorful, ellipsoid patch is attached next to the lesion to mark the lesion as malignant. Do you see a colorful, ellipsoid patch attached on the skin? Answer with one word of the options [yes, no]"
          #"The image shows a lesion and sometimes a coloful ellipsoid patch is placed next to it. The patch does not belong to the skin. Does this image has a patch? Answer with yes or no, with one word.",
          #"The image shows part of a skin with a lesion. Sometimes a coloful ellipsoid patch is placed next to the lesion to mark if it is malignant. The patch does not belong to the skin. Do you see a colorful, ellipsoid patch attached on the skin. Answer with one word of the options [yes, no].",
          "The image shows part of a skin with a lesion. Sometimes a colorful ellipsoid patch is placed next to the lesion to mark if it is malignant. The patch does not belong to the skin. Do you see a colorful, ellipsoid patch attached on the skin or not? Answer with one word of the options [yes, no].", 
          
          #"The image shows a part of skin with a lesion. Often a patch marks if it is cancer. I want to figure out if this image has such a patch. Do you see a patch? Please answer with yes or no. Please answer with one word.",
          #"Is a patch in the image? Please answer with one word, yes or no.",
          #"Is a patch in the image? Answer with yes or no.",
          #"The image shows a part of skin with a lesion. Often a patch marks if it is cancer. I want to figure out if this image has such a patch. Do you see a patch? Please answer with one word, yes or no.",
          #"Can you see a patch?",
          #"Can you see a patch? Reply with one word, yes or no.",
          #"patch?",
          #"patch? One word reply."
         ]


for prompt in prompts:
    run_inference(images, num_imgs=100, question1=prompt, question2=None, data_dict=data_dict)
    